# Matrix Operations — Multivectors Meet NumPy

**Part II · Geometric Algebra** — Tutorial 12

This tutorial bridges geometric algebra and linear algebra. `MVMatrix` wraps a
NumPy array whose rows are labelled by a `BladeMask`; `MVProductMatrix` encodes a
GA product as a 3-D tensor of matrices. These types are the machinery behind the
equation solvers from [Tutorial 11](../11_equation_solving/).

By the end you will be able to:

- Store multivector coefficients in an `MVMatrix` with labelled rows.
- Convert between `MV` and NumPy arrays with `to_matrix` / `from_matrix`.
- Batch multiple multivectors as columns of one matrix.
- Build a product matrix with `product_matrix()`.
- Verify that a product matrix encodes a GA product as a linear map.
- Read the reverse/conjugate sign matrices with `product_matrix_rev` /
  `product_matrix_conj`.

> **Prerequisites:** [Tutorial 10](../10_blade_mask/) (blade masks) and
> [Tutorial 11](../11_equation_solving/) (the solver pipeline).

## 1. Setup

The matrix types live in `pytanga.matrix`, with conversion and product-matrix
helpers in `pytanga.matrix.convert` and `pytanga.matrix.product`.

In [1]:
import numpy as np

from pytanga import BladeMask
from pytanga.basis import BasisE3
from pytanga.matrix import MVMatrix, MVProductMatrix
from pytanga.matrix.convert import from_matrix, to_matrix
from pytanga.matrix.product import product_matrix, product_matrix_conj, product_matrix_rev

E3 = BasisE3()
full = BladeMask.full(E3)     # all 8 blades: s, e1, e2, e12, e3, e13, e23, I

## 2. `MVMatrix` — a row-labelled coefficient matrix

`MVMatrix` is a small dataclass pairing a 2-D NumPy array with a `BladeMask` that
labels its **rows**. Each column stores the coefficient vector of one multivector,
ordered by `row_mask.ids`.

In [2]:
m = MVMatrix(data=np.zeros((8, 1)), row_mask=full)

print("shape     :", m.shape)
print("n_cols    :", m.n_cols)
print("is_single :", m.is_single)
print("algebra   :", type(m.algebra).__name__)
print("row_mask  :", m.row_mask.names())

shape     : (8, 1)
n_cols    : 1
is_single : True
algebra   : BasisE3
row_mask  : ['s', 'e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'I']


## 3. `to_matrix` / `from_matrix`

`to_matrix(mv, mask=...)` fills a column vector from a multivector's coefficients
(in `mask.ids` order); `from_matrix(mat)` reconstructs the multivector(s).

In [3]:
mv = E3("2 e1 - 3 e2 + 4 e12")

mat = to_matrix(mv, mask=full)
print("data shape     :", mat.data.shape)
print("coefficients   :", mat.data[:, 0].tolist())
print("round-trip MV  :", from_matrix(mat).to_dict())

data shape     : (8, 1)
coefficients   : [0.0, 2.0, -3.0, 4.0, 0.0, 0.0, 0.0, 0.0]
round-trip MV  : {'e1': 2.0, 'e2': -3.0, 'e12': 4.0}


## 4. Batching — many multivectors as columns

Pass a list of multivectors to `to_matrix` to stack them into an
`(n_rows, n_mvs)` matrix. `from_matrix` then returns a list of `MV`s.

In [4]:
batch = to_matrix([E3("e1"), E3("e2"), E3("e12")], mask=full)

print("batch shape :", batch.data.shape)
print("n_cols      :", batch.n_cols)
print("is_single   :", batch.is_single)

mvs = from_matrix(batch)
print("round-trip   :", [x.to_dict() for x in mvs])

batch shape : (8, 3)
n_cols      : 3
is_single   : False
round-trip   : [{'e1': 1.0}, {'e2': 1.0}, {'e12': 1.0}]


## 5. `product_matrix` — building the linear map

`product_matrix(A, a_mask=..., b_mask=..., c_mask=...)` returns an
`MVProductMatrix`: a 3-D tensor of shape `(|a_mask|, |c_mask|, |b_mask|)`. Each
slice `M.data[i]` is the `(|c_mask| × |b_mask|)` matrix that maps the coefficients
of `X` to the coefficients of `A_i ∘ X`.

In [5]:
vecs = BladeMask(E3, grades=[1])       # unknown X lives in the vector subspace

M = product_matrix(
    E3("e1"),
    a_mask=BladeMask(E3, "e1"),
    b_mask=vecs,
    c_mask=full,
)

print("n_mvs    :", M.n_mvs)
print("shape    :", M.shape)          # (1, 8, 3)
print("a / b / c:", M.a_mask.ids, M.b_mask.ids, M.c_mask.ids)
print("product  :", M.product, " left:", M.left)

n_mvs    : 1
shape    : (1, 8, 3)
a / b / c: [1] [1, 2, 4] [0, 1, 2, 3, 4, 5, 6, 7]
product  : gp  left: True


## 6. The product matrix encodes a GA product

For a single `A` (`|a_mask| == 1`), `M.data[0] @ vec(X) == vec(A ∘ X)`. We verify
it for `A = e1` and `X = 2 e2`: the geometric product is `2 e12`.

In [6]:
A = E3("e1")
X = E3("2 e2")

v = to_matrix(X, mask=vecs)                    # X coefficients in {e1,e2,e3} order
result_vec = np.matmul(M.data[0], v.data)       # (8, 3) @ (3, 1) -> (8, 1)
result_mv = from_matrix(MVMatrix(result_vec, full))

print("vec(X)      :", v.data[:, 0].tolist())
print("M @ vec(X)  :", result_mv.to_dict())
print("A * X       :", (A * X).to_dict())

vec(X)      : [0.0, 2.0, 0.0]
M @ vec(X)  : {'e12': 2.0}
A * X       : {'e12': 2.0}


## 7. Reverse and conjugate matrices

`product_matrix_rev(mask)` and `product_matrix_conj(mask)` return square diagonal
matrices whose diagonal entries are the reverse / conjugate signs of each blade:
grade `k` contributes `(-1)^(k(k-1)/2)` (reverse), plus the negative-metric sign
for the conjugate. In Euclidean E3 they coincide.

In [7]:
R = product_matrix_rev(full)
C = product_matrix_conj(full)

print("blades      :", full.names())
print("reverse diag:", np.diag(R.data[0]).astype(int).tolist())
print("conj    diag:", np.diag(C.data[0]).astype(int).tolist())

blades      : ['s', 'e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'I']
reverse diag: [1, 1, 1, -1, 1, -1, -1, -1]
conj    diag: [1, 1, 1, -1, 1, -1, -1, -1]


## 8. Summary & next steps

| Task | API |
|---|---|
| Row-labelled coefficient matrix | `MVMatrix(data, row_mask)` |
| MV → NumPy column(s) | `to_matrix(mv, mask=...)`, `to_matrix([...], mask=...)` |
| NumPy column(s) → MV | `from_matrix(mat)` |
| Product matrix | `product_matrix(A, a_mask=..., b_mask=..., c_mask=...)` |
| Product tensor axes | `.a_mask`, `.b_mask`, `.c_mask` |
| Reverse / conjugate matrix | `product_matrix_rev(mask)`, `product_matrix_conj(mask)` |

**Where to go next:**

- [**13 · Tensor Operations**](../13_tensor/) — `MVTensor` and label-driven
  contractions built on top of these masks.
- [**11 · Equation Solving**](../11_equation_solving/) — how the solvers consume
  `product_matrix`.